In [ ]:
# !/usr/bin/env python
#
# Script info
# -----------
# __author__ = 'Ryan Godin'
# __copyright__ = '© His Majesty the King in Right of Canada, as represented by the Minister of Agriculture and Agri-Foods Canada, 2017-2025'
# __credits__ = 'Ryan Godin, Etienne Lord'
# __email__ = ''
# __license__ = 'Open Government Licence - Canada'
# __maintainer__ = 'Ryan Godin'
# __status__ = 'Development'
# __version__ = '1.1.1'

"""
t-Distributed Stochastic Neighbor Embedding (t-SNE) Script

Recent Changes:
>>> Added support for Sentinel-1 channels ('VV', 'VH', 'VV_VH_Ratio').
>>> Added balanced sampling to ensure equal representation of classes.
>>> Improved data normalization using StandardScaler.

Example Usage:
>>> filepath = 'path/to/dataset.npz'
>>> tsne = tSNE(filepath, max_per_class=100)
"""

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

In [ ]:
class tSNE:
    def __init__(self, filepath, max_per_class=None, random_state=None):
        """
        Initializes the t-SNE visualization utility.
        Parameters:
            filepath (str): Path to the .npz file containing the dataset. The file must have the keys:
                - 'data': numpy array of shape (N, CHANNELS, HEIGHT, WIDTH) or similar.
                - 'labels': array of integer class labels.
                - 'crops': array of class label names corresponding to the integer labels.
            max_per_class (int or None, optional): If specified, limits the number of samples per class to this value,
                ensuring a balanced sample across classes. If None, uses all available samples.
            random_state (int or None, optional): Seed for random number generator to ensure reproducibility.
                If None, uses the current timestamp.
        Loads and preprocesses the data, performs balanced sampling if requested, normalizes features,
        applies t-SNE for dimensionality reduction, and prepares a color map for visualization.
        """

        # Load Data
        self.filepath = filepath
        self.data = np.load(filepath, allow_pickle=True)

        self.x = self.data['data']     # (N, CHANNELS, HEIGHT, WIDTH)
        self.y = self.data['labels']

        # Reshape to 2D (n_samples, n_features)
        if self.x.ndim > 2:
            n_samples = self.x.shape[0]
            n_features = np.prod(self.x.shape[1:])
            self.x = self.x.reshape(n_samples, n_features)

        # Labels
        self.crop_labels = self.data['crops']
        self.label_map = {i: name for i, name in enumerate(self.crop_labels)}

        valid_y_indices = np.isin(self.y, list(self.label_map.keys()))
        self.labels = np.array([self.label_map[label] for label in self.y[valid_y_indices]])
        self.x = self.x[valid_y_indices]

        # Color map
        unique_labels = np.unique(self.labels)
        cmap = plt.colormaps.get_cmap('jet')
        colors = cmap(np.linspace(0, 1, len(unique_labels)))
        self.color_map = {label: colors[i] for i, label in enumerate(unique_labels)}

        # RNG
        self.time = int(time.time()) if random_state is None else int(random_state)
        self.rng = np.random.default_rng(self.time)

        # -------- Balanced sampling --------
        if self.x.shape[0] > 0:
            if max_per_class is None:
                self.indices = np.arange(self.x.shape[0])
            else:
                self.indices = self._balanced_sample_indices(self.labels, max_per_class)

            if self.indices.size == 0:
                print('[WARN] No data points selected after balanced sampling.')
                return

            self.sampled_data = self.x[self.indices]
            self.sampled_labels = self.labels[self.indices]

            # --- Normalize ALL channels the same way (mean=0, std=1) ---
            scaler = StandardScaler()
            self.sampled_data = scaler.fit_transform(self.sampled_data)

            # Run t-SNE
            tsne = TSNE(n_components=2, random_state=self.time)
            self.sampled_data_tsne = tsne.fit_transform(self.sampled_data)

            # Plot
            self._plot()
        else:
            print('[WARN] Not enough data points to sample for t-SNE.')

    def _balanced_sample_indices(self, labels, max_per_class):
        """
        Selects a balanced subset of sample indices from the input labels, with a maximum number of samples per class.

        Parameters:
            labels (np.ndarray): Array of class labels for each sample.
            max_per_class (int): Maximum number of samples to select per class.

        Returns:
            np.ndarray: Array of selected indices, shuffled, containing at most `max_per_class` samples per unique class label.
        """
        idx_list = []
        for lab in np.unique(labels):
            lab_idx = np.where(labels == lab)[0]
            k = min(max_per_class, lab_idx.size)
            if k > 0:
                chosen = self.rng.choice(lab_idx, size=k, replace=False)
                idx_list.append(chosen)
        if not idx_list:
            return np.array([], dtype=int)
        all_idx = np.concatenate(idx_list)
        self.rng.shuffle(all_idx)
        return all_idx

    def _plot(self):
        plt.figure(figsize=(8, 6))
        if self.sampled_labels.size > 0:
            for label in np.unique(self.sampled_labels):
                label_indices = (self.sampled_labels == label)
                plt.scatter(
                    self.sampled_data_tsne[label_indices, 0],
                    self.sampled_data_tsne[label_indices, 1],
                    label=label,
                    c=[self.color_map[label]],
                    alpha=0.8,
                    edgecolors='w',
                    linewidths=0.5,
                    s=24
                )
            plt.title('t-SNE Visualization of Cropland Dataset')
            plt.xlabel('Axis 1')
            plt.ylabel('Axis 2')
            plt.legend(markerscale=1.2, frameon=True)
            plt.tight_layout()
            plt.savefig(f'tSNE_FILENAME_{self.time}.svg')
            plt.show()
        else:
            print('[WARN] No data points to plot after sampling.')

### Example Usage

In [ ]:
filepath = r'path/to/dataset.npz'
tsne = tSNE(filepath, max_per_class=100)